# Fine-tune Perception Encoder (RSTPReid + TVPReid train)

**PE-B/16** (Run A) or **PE-L/14** (Run B) for text-to-tracklet retrieval. Both runs train **last vision blocks + text** with a **frozen logit_scale** — full-tower FT overfits within epoch 0 on PEDES-style data (not QLoRA).

| Data | Split | Role |
|---|---|---|
| **RSTPReid** | official train + val | still-image person captions |
| **TVPReid** (`prid`, `ilids`, `duke`) | official train + val | videos → uniform still frames |

**Never** trains on Alexandria, GroOT, or any test split. After training, download `best.pt` and eval locally with `scripts/eval_finetuned.py` (Alexandria + GroOT stay out-of-domain).

**How to run**

1. GPU T4 (or 2×T4), **Internet on**.
2. Optional: Kaggle secret `hugging_face` (helps Hub downloads / rate limits).
3. Set `RUN = "A"` or `"B"` in the config cell, then **Run All**.
4. First session downloads RSTPReid (Google Drive) + TVPReid train/val mp4s (Hugging Face), then packages frames and trains.

## 1. Configuration

In [ ]:
from pathlib import Path

RUN = "A"  # "A" = PE-B/16 last-6 blocks+text; "B" = PE-L/14 last-4 (or LoRA)
USE_LORA = False  # Run B only; True = LoRA (fp16), not QLoRA
UNFREEZE_LAST_BLOCKS = 6 if RUN == "A" else 4  # None = full FT (overfits fast)
FREEZE_LOGIT_SCALE = True
EPOCHS = 5 if RUN == "A" else 8
BATCH = 64 if RUN == "A" else 16
LEARNING_RATE = 2e-6 if RUN == "A" else 1e-6
EARLY_STOP_PATIENCE = 2
MAX_STEPS = None  # e.g. 50 for a smoke run; None = full epochs
FRAMES_PER_VIDEO = 4
TVPREID_SUBSETS = ["prid", "ilids", "duke"]
DEVICE = "cuda"

# Branch with the fine-tune stack. After merge to main, set to "main".
SHAWAF_BRANCH = "feat/alexandria-ingestion-benchmark"

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("results/finetune")
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "finetune_data"
OUT_DIR = WORK / ("pe_b16_run_a" if RUN == "A" else "pe_l14_run_b")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"Run {RUN} epochs={EPOCHS} batch={BATCH} lr={LEARNING_RATE}",
    f"branch={SHAWAF_BRANCH}",
    f"out={OUT_DIR}",
    flush=True,
)

## 2. Setup

Clones the shawaf repo (same pattern as the zero-shot notebook), installs `[all]` + `gdown`, checks the package version, and optionally loads the Hugging Face token.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
EXPECTED = "0.1.18"


def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.check_call([str(p) for p in cmd], cwd=str(cwd) if cwd else None)


def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def sync_shawaf(url, path, branch):
    if path.exists():
        run(["git", "fetch", "origin"], cwd=path)
        run(["git", "checkout", branch], cwd=path)
        run(["git", "reset", "--hard", f"origin/{branch}"], cwd=path)
    else:
        run(["git", "clone", "--branch", branch, url, str(path)])


ON_KAGGLE = Path("/kaggle/working").is_dir()

if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    sync_shawaf(SHAWAF_URL, shawaf_dir, SHAWAF_BRANCH)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]", "gdown", "opencv-python-headless"])
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break

if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))

for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("open_clip", installed("open_clip_torch"), "torch", installed("torch"), flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and re-run setup after the branch updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if ON_KAGGLE and not token:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("hugging_face")
    except Exception as exc:
        print("No hugging_face secret:", type(exc).__name__, flush=True)
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login

    login(token=token, add_to_git_credential=False)
    print("Hugging Face token loaded", flush=True)
else:
    print("Continuing without HF token (public Hub downloads)", flush=True)

ROOT = shawaf_dir
print("ROOT", ROOT, flush=True)

## 3. Datasets

Same download helpers as the zero-shot notebook:

- **RSTPReid** — Google Drive archive via `gdown` (or a Kaggle input that already has `data_captions.json` + `imgs/`).
- **TVPReid** — public Hugging Face `bassatbassat/TVPReid`, **train** and **val** mp4s only (not test).

TVPReid train is ~5k small files. Hugging Face rate-limits that volume (HTTP 429) — that is **not** a private-repo error. The downloader uses few workers, disables Xet, cools down on 429, and **resumes** if you re-run the cell.

Then packages into `finetune_data/<source>/{train,val}.jsonl` + resized stills.

In [ ]:
import os

from shawaf_vlm.data.rstpreid import download_rstpreid
from shawaf_vlm.data.tvpreid import download_tvpreid

# Public Hub dataset — 429s are rate limits from many file requests, not access.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Downloading RSTPReid…", flush=True)
RSTP_ROOT = download_rstpreid()
print("RSTPReid", RSTP_ROOT, flush=True)

# Train is ~5k mp4s. Download one subset at a time, 2 workers max; re-run resumes.
print("Downloading TVPReid train + val (slow + resumable; ignore transient 429s)…", flush=True)
for split in ("train", "val"):
    for subset in TVPREID_SUBSETS:
        print(f"TVPReid {split}/{subset}…", flush=True)
        TVP_ROOT = download_tvpreid(
            configs=(subset,),
            split=split,
            max_workers=2,
        )
        print(f"TVPReid {split}/{subset} done → {TVP_ROOT}", flush=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "prepare_finetune_kaggle.py"),
    "--out",
    str(DATA),
    "--frames-per-video",
    str(FRAMES_PER_VIDEO),
]
print("+", " ".join(cmd), flush=True)
subprocess.check_call(cmd)

train_files = sorted(DATA.glob("*/train.jsonl"))
val_files = sorted(DATA.glob("*/val.jsonl"))
print("train", [p.name + " @ " + p.parent.name for p in train_files], flush=True)
print("val", [p.name + " @ " + p.parent.name for p in val_files], flush=True)
assert any(p.parent.name == "rstpreid" for p in train_files), "RSTPReid train.jsonl missing"
assert any(p.parent.name == "tvpreid" for p in train_files), "TVPReid train.jsonl missing"

for path in train_files:
    n = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"  {path.parent.name}/train.jsonl rows={n}", flush=True)

## 4. Train

Checkpoints land in `/kaggle/working/pe_b16_run_a` (or `pe_l14_run_b`). Only **`last.pt`** + **`best.pt`** are kept (no per-epoch dumps / optimizer — Kaggle disk fills fast otherwise).

If a previous run died on `iostream` / `unexpected pos` while saving: free space (delete `epoch_*.pt`), pull latest code, set `RESUME` to `last.pt` or `best.pt`, re-run. Download **`best.pt`** when finished.

Note: rising `val_loss` while train drops is expected early overfitting on PEDES-style data; **`best.pt`** tracks lowest val.

In [ ]:
import shutil
from pathlib import Path

from shawaf_vlm.finetune.checkpoint import free_disk_gb
from shawaf_vlm.finetune.train import FinetuneConfig, train_pe

# If the last run crashed mid-save: clear fat epoch dumps, then optionally resume.
RESUME = None  # e.g. str(OUT_DIR / "last.pt") or str(OUT_DIR / "best.pt")
for junk in OUT_DIR.glob("epoch_*.pt"):
    junk.unlink(missing_ok=True)
    print("removed", junk, flush=True)
for junk in OUT_DIR.glob("*.pt.tmp"):
    junk.unlink(missing_ok=True)

free = free_disk_gb(WORK)
print(f"free disk under {WORK}: {free:.1f} GiB" if free is not None else "disk free: unknown", flush=True)

if RUN == "A":
    cfg = FinetuneConfig(
        base_key="pe_core_b16",
        output_dir=str(OUT_DIR),
        epochs=EPOCHS,
        batch_size=BATCH,
        lr=LEARNING_RATE,
        image_size=224,
        fit="letterbox",
        device=DEVICE,
        unfreeze_last_blocks=UNFREEZE_LAST_BLOCKS,
        use_lora=False,
        freeze_logit_scale=FREEZE_LOGIT_SCALE,
        save_every_epoch=False,
        save_optimizer=False,
        early_stop_patience=EARLY_STOP_PATIENCE,
        resume=RESUME,
    )
else:
    cfg = FinetuneConfig(
        base_key="pe_core_l14",
        output_dir=str(OUT_DIR),
        epochs=EPOCHS,
        batch_size=BATCH,
        lr=LEARNING_RATE,
        image_size=336,
        fit="letterbox",
        device=DEVICE,
        unfreeze_last_blocks=None if USE_LORA else UNFREEZE_LAST_BLOCKS,
        use_lora=USE_LORA,
        freeze_logit_scale=FREEZE_LOGIT_SCALE,
        grad_checkpoint=True,
        save_every_epoch=False,
        save_optimizer=False,
        early_stop_patience=EARLY_STOP_PATIENCE,
        resume=RESUME,
    )

cfg.train_jsonl = [str(p) for p in train_files]
cfg.val_jsonl = [str(p) for p in val_files]
cfg.image_root = str(DATA)
cfg.max_steps = MAX_STEPS
cfg.num_workers = 2
cfg.log_every = 50

print(cfg, flush=True)
best = train_pe(cfg)
env = "SHAWAF_PE_B16_FT" if cfg.base_key == "pe_core_b16" else "SHAWAF_PE_L14_FT"
model_key = f"{cfg.base_key}_ft"
print("best checkpoint:", best, flush=True)
print(
    "Local eval after download:\n"
    f"  set {env}={best.name}\n"
    f"  python scripts/eval_finetuned.py --model {model_key} --fit letterbox --max-frames 16",
    flush=True,
)